In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, log_loss
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)
print('imports OK')

---
## Problem 8: Contextual Bandits (LinUCB)

### Business Context
A/B testing is slow and wasteful — it allocates equal traffic to all variants including poor ones. Bandits adaptively allocate more traffic to better-performing options while still exploring.

### Exploration-Exploitation Trade-off
- **Exploit**: choose the action with highest estimated reward (greedy)
- **Explore**: try less-certain actions to gather more information
- **Regret**: cumulative loss vs. always choosing the best action

### Algorithms

| Algorithm | Exploration Strategy | Context-Aware | Best For |
|-----------|---------------------|---------------|----------|
| Epsilon-greedy | Random (ε fraction) | No | Simple, robust baseline |
| UCB1 | Upper confidence bound | No | Stationary rewards |
| Thompson Sampling | Posterior sampling | Yes (with context) | Correlated arms |
| LinUCB | Linear UCB with context | Yes | Marketing personalization |

### LinUCB
Assumes reward is linear in context: r = θ_a^T x + noise.  
Upper confidence bound: UCB_a(x) = θ_a^T x + α √(x^T A_a^{-1} x)  
- First term: exploitation (estimated reward)
- Second term: exploration bonus (uncertainty of estimate for this context)

In [ ]:
class LinUCB:
    """
    LinUCB contextual bandit.
    Each arm has its own ridge regression model.
    """
    def __init__(self, n_arms, context_dim, alpha=1.0):
        self.n_arms = n_arms
        self.alpha  = alpha          # exploration strength
        # Per-arm: A_a (covariance), b_a (reward accumulator)
        self.A = [np.eye(context_dim) for _ in range(n_arms)]
        self.b = [np.zeros(context_dim) for _ in range(n_arms)]

    def select_arm(self, context):
        """Choose arm with highest UCB score."""
        ucb_scores = []
        for a in range(self.n_arms):
            A_inv = np.linalg.inv(self.A[a])
            theta = A_inv @ self.b[a]                         # estimated reward weights
            exploit = theta @ context                          # estimated reward
            explore = self.alpha * np.sqrt(context @ A_inv @ context)  # uncertainty
            ucb_scores.append(exploit + explore)
        return int(np.argmax(ucb_scores))

    def update(self, arm, context, reward):
        """Update arm's model with observed reward."""
        self.A[arm] += np.outer(context, context)
        self.b[arm] += reward * context

# Simulation: 3 arms (ad variants), context = user features
n_arms, context_dim, T = 3, 5, 1000
true_theta = np.random.randn(n_arms, context_dim)   # unknown true reward weights

bandit = LinUCB(n_arms, context_dim, alpha=0.5)
cumulative_regret = 0
regrets = []

for t in range(T):
    context = np.random.randn(context_dim)

    # Oracle: best arm for this context
    true_rewards = true_theta @ context
    best_arm = np.argmax(true_rewards)

    # LinUCB selects arm
    chosen = bandit.select_arm(context)

    # Observe reward (noisy)
    reward = true_theta[chosen] @ context + np.random.randn() * 0.1

    # Update model
    bandit.update(chosen, context, reward)

    # Track regret
    cumulative_regret += true_rewards[best_arm] - true_rewards[chosen]
    regrets.append(cumulative_regret)

print(f'LinUCB simulation over {T} steps:')
print(f'  Final cumulative regret: {cumulative_regret:.4f}')
print(f'  Avg regret per step:     {cumulative_regret/T:.4f}')

# Compare arm selection distribution
arm_counts = defaultdict(int)
for t in range(200):  # final 200 steps
    context = np.random.randn(context_dim)
    arm_counts[bandit.select_arm(context)] += 1
print(f'\nArm selection (last 200 steps): {dict(arm_counts)}')

### L6 Interview Q&A

**Q: When would you use a bandit instead of an A/B test?**  
A: Use bandits when: (1) the cost of showing a bad variant is high (revenue loss, user churn), (2) many variants need testing simultaneously, (3) the optimal action changes over time (non-stationary rewards). Use A/B when: you need a clean causal estimate for a business report, regulatory requirement, or the exploration budget is short.

**Q: What's the difference between UCB1 and LinUCB?**  
A: UCB1 is context-free — it estimates a single reward distribution per arm from all observations. LinUCB is context-aware — it models reward as a linear function of user features. LinUCB adapts the exploration bonus based on how uncertain it is about THIS specific user's context, not just overall arm uncertainty. In a marketing setting, LinUCB can learn that arm A is better for young urban users while arm B is better for suburban users.

**Q: How do you handle non-stationary rewards (user preferences change over time)?**  
A: Options: (1) Sliding window — use only recent observations (discard old A and b matrices), (2) Discounted updates — weight recent observations more heavily, (3) Change-point detection — detect when reward distribution shifts and reset the model, (4) Periodic full retraining.

### Common Pitfalls
- Using bandit with too high alpha — over-exploration, converges slowly
- Forgetting that bandits require online serving infrastructure (A matrix must be updated in real time)
- Mixing bandit traffic with A/B holdout traffic — confounds the causal estimate